# Model privacy and memorization audit

This notebook tests whether an AspectBench release exposes an obvious membership or entity-memorization signal. It is a **risk screen, not a proof of privacy**. The models are discriminative classifiers, so their normal interface cannot emit arbitrary training articles, but model inversion, membership inference, and property inference remain relevant threat models.

The notebook never writes article text, aspect names, per-record probabilities, or Slovenian examples. Only aggregate statistics and checkpoint metadata are saved. Run HBS first. Slovenian execution requires an explicit environment flag and its output remains in the ignored `outputs/` tree.

## Interpretation boundaries

- A checkpoint containing tensors only rules out accidental packaging of corpus rows; it does not rule out memorization in parameters.
- Membership AUC near 0.5 is reassuring only for the tested attack and cohort construction.
- Validation examples influenced early stopping and split selection, so validation-based nonmember results can be conservative or biased. The test-set sensitivity analysis helps, but distribution shift can also inflate AUC.
- Entity-name counterfactual sensitivity measures reliance on target identity, not verbatim extraction. High sensitivity deserves qualitative investigation but is not automatically a privacy breach.
- Formal differential privacy was not used; this notebook must not claim a mathematical privacy guarantee.

In [1]:
from __future__ import annotations

from collections import Counter
from datetime import datetime, timezone
import gc
import hashlib
import json
import math
import os
from pathlib import Path
import re
import sys

import numpy as np
import pandas as pd

REPO_ROOT = Path(os.environ.get('ASPECTBENCH_ROOT', '.')).resolve()
if not (REPO_ROOT / 'src' / 'aspectbench').is_dir():
    REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT / 'src'))

from aspectbench.inference.hf_bridge import create_engine
from aspectbench.privacy import checkpoint_tensor_inventory, membership_attack_report

DATASET = os.environ.get('PRIVACY_DATASET', 'hbs')  # hbs or sl
MODELS = os.environ.get('PRIVACY_MODELS', 'xlmr,han-xlmr,bge-m3-mlp').split(',')
VARIANT = os.environ.get('PRIVACY_VARIANT', 'unmasked')
DEVICE = os.environ.get('PRIVACY_DEVICE', 'cuda')
MAX_PER_COHORT = int(os.environ.get('PRIVACY_MAX_PER_COHORT', '384'))
BOOTSTRAPS = int(os.environ.get('PRIVACY_BOOTSTRAPS', '1000'))
PERMUTATIONS = int(os.environ.get('PRIVACY_PERMUTATIONS', '1000'))
SEED = int(os.environ.get('PRIVACY_SEED', '42'))
RUN_MODEL_ATTACKS = os.environ.get('RUN_MODEL_ATTACKS', '0') == '1'
RUN_COUNTERFACTUALS = os.environ.get('RUN_COUNTERFACTUALS', '0') == '1'
ALLOW_RESTRICTED_SLOVENE = os.environ.get('ALLOW_RESTRICTED_SLOVENE', '0') == '1'
RUN_ID = os.environ.get('PRIVACY_RUN_ID', datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
OUTPUT_DIR = REPO_ROOT / 'outputs' / 'privacy-audit' / DATASET / RUN_ID

if DATASET not in {'hbs', 'sl'}:
    raise ValueError('PRIVACY_DATASET must be hbs or sl')
if DATASET == 'sl' and not ALLOW_RESTRICTED_SLOVENE:
    raise RuntimeError('Set ALLOW_RESTRICTED_SLOVENE=1 only in an authorized internal environment.')
if VARIANT not in {'masked', 'unmasked'}:
    raise ValueError('PRIVACY_VARIANT must be masked or unmasked')

LANGUAGE = 'slovenian' if DATASET == 'sl' else 'hbs'
DATA_PREFIX = 'slovene' if DATASET == 'sl' else 'hbs'
MODEL_ROOT = REPO_ROOT / 'huggingface' / 'models'
print({'dataset': DATASET, 'models': MODELS, 'variant': VARIANT, 'device': DEVICE,
       'run_model_attacks': RUN_MODEL_ATTACKS, 'run_counterfactuals': RUN_COUNTERFACTUALS})

{'dataset': 'hbs', 'models': ['xlmr', 'han-xlmr', 'bge-m3-mlp'], 'variant': 'unmasked', 'device': 'cuda', 'run_model_attacks': False, 'run_counterfactuals': False}


## 1. Artifact-level release audit

This confirms that each selected checkpoint is a flat tensor state dictionary. Optimizer checkpoints and training directories must never be uploaded. For BGE-M3, the released file is only the learned MLP head; the frozen `BAAI/bge-m3` encoder is fetched separately. XLM-R and HAN-XLM-R files contain their complete fine-tuned encoders and classifier-specific parameters.

In [2]:
inventories = []
for model in MODELS:
    checkpoint = MODEL_ROOT / model / LANGUAGE / f'{VARIANT}.pt'
    report = checkpoint_tensor_inventory(checkpoint)
    report['model'] = model
    report['language'] = LANGUAGE
    report['variant'] = VARIANT
    inventories.append(report)
    gc.collect()

inventory_frame = pd.DataFrame(inventories)
display(inventory_frame[['model', 'size_bytes', 'tensor_count', 'parameter_count',
                         'dtypes', 'tensor_only', 'sha256']])
assert inventory_frame['tensor_only'].all(), 'A release checkpoint contains non-tensor payloads.'

,model,size_bytes,tensor_count,parameter_count,dtypes,tensor_only,sha256
0,xlmr,1112263649,201,278045955,[torch.float32],True,a6a1e25a048de9c4356e767d705cd8639406348772b870...
1,han-xlmr,1179611454,232,294879235,[torch.float32],True,7bfea9a485b517fee669b343e6c382f1c927a8a06931cb...
2,bge-m3-mlp,2630266,6,656899,[torch.float32],True,67a0e1b42f74df9c391e2e0d6697341130576abb4f1aed...


## 2. Construct comparable member and nonmember cohorts

For each released model, members come from the training partition of the split that produced that checkpoint. Primary nonmembers come from that split's validation partition; the test set is a sensitivity cohort. Sampling is balanced by sentiment class and capped without printing any record. Duplicate hashed records across cohorts are rejected.

In [3]:
def load_json_rows(path: Path, key: str):
    payload = json.loads(path.read_text(encoding='utf-8'))
    rows = payload[key]
    if not rows:
        raise ValueError(f'No {key} rows in {path}')
    return rows

def privacy_id(row):
    material = f"{row.get('uuid', '')}\0{row.get('article', '')}\0{row.get('aspect', '')}"
    return hashlib.sha256(material.encode('utf-8')).hexdigest()[:24]

def balanced_sample(rows, limit, seed):
    rng = np.random.default_rng(seed)
    groups = {label: [row for row in rows if int(row['sentiment']) == label]
              for label in (-1, 0, 1)}
    per_class = min(max(1, limit // 3), *(len(group) for group in groups.values()))
    sampled = []
    for label, group in groups.items():
        indices = rng.choice(len(group), per_class, replace=False)
        sampled.extend(group[int(index)] for index in indices)
    rng.shuffle(sampled)
    return sampled

def selected_split(model):
    metadata = json.loads((MODEL_ROOT / model / 'availability.json').read_text())
    match = next(row for row in metadata['entries']
                 if row['language'] == LANGUAGE and row['mode'] == VARIANT)
    split = match.get('selected_split')
    if split is None:
        manifest = json.loads((MODEL_ROOT / 'manifest.json').read_text())
        split = next(row['run'] for row in manifest['entries']
                     if row['model'] == model and row['language'] == LANGUAGE
                     and row['mode'] == VARIANT)
    return int(split)

def cohorts_for(model):
    split = selected_split(model)
    split_path = REPO_ROOT / 'data' / DATASET / f'{DATA_PREFIX}_train_val_{split}.json'
    payload = json.loads(split_path.read_text(encoding='utf-8'))
    members = balanced_sample(payload['train'], MAX_PER_COHORT, SEED + split)
    validation = balanced_sample(payload['val'], MAX_PER_COHORT, SEED + 100 + split)
    test_path = REPO_ROOT / 'data' / DATASET / f'{DATA_PREFIX}_test.json'
    test = balanced_sample(load_json_rows(test_path, 'test'), MAX_PER_COHORT, SEED + 200 + split)
    member_ids = {privacy_id(row) for row in members}
    if member_ids & {privacy_id(row) for row in validation + test}:
        raise RuntimeError('Member/nonmember record overlap detected.')
    return split, members, validation, test

cohort_summary = []
for model in MODELS:
    split, members, validation, test = cohorts_for(model)
    cohort_summary.append({'model': model, 'selected_split': split,
                           'members': len(members), 'validation_nonmembers': len(validation),
                           'test_nonmembers': len(test)})
display(pd.DataFrame(cohort_summary))

,model,selected_split,members,validation_nonmembers,test_nonmembers
0,xlmr,0,384,384,384
1,han-xlmr,1,384,384,384
2,bge-m3-mlp,1,384,384,384


## 3. Black-box membership inference

The attacks use information already returned by ordinary inference: gold-label log probability, maximum confidence, negative predictive entropy, and top-two margin. Higher scores mean ‘more likely a member’. Enable with `RUN_MODEL_ATTACKS=1`; a GPU is strongly recommended. Only aggregate AUC, bootstrap intervals, permutation p-values, and maximum TPR−FPR are retained.

In [4]:
CLASS_KEY = {-1: '-1 (negative)', 0: '0 (neutral)', 1: '1 (positive)'}

def safe_scores(outputs, source_rows):
    safe = []
    for output, source in zip(outputs, source_rows, strict=True):
        probs = np.array([output['class_probabilities'][CLASS_KEY[label]]
                          for label in (-1, 0, 1)], dtype=float)
        gold_index = int(source['sentiment']) + 1
        entropy = -float(np.sum(np.clip(probs, 1e-12, 1) *
                                np.log(np.clip(probs, 1e-12, 1))))
        ordered = np.sort(probs)
        safe.append({
            'gold_log_probability': float(np.log(max(probs[gold_index], 1e-12))),
            'confidence': float(probs.max()),
            'negative_entropy': -entropy,
            'margin': float(ordered[-1] - ordered[-2]),
            'correct': int(probs.argmax() == gold_index),
            'label': int(source['sentiment']),
        })
    return safe

def infer_scores(engine, rows):
    batch_size = 1 if engine.backend == 'han' else 8
    outputs = engine.predict_batch(rows, batch_size=batch_size, mc_passes=0, seed=SEED)
    return safe_scores(outputs, rows)

def compare_cohorts(member_rows, nonmember_rows):
    reports = {}
    for feature in ('gold_log_probability', 'confidence', 'negative_entropy', 'margin'):
        reports[feature] = membership_attack_report(
            [row[feature] for row in member_rows],
            [row[feature] for row in nonmember_rows],
            seed=SEED, bootstrap_samples=BOOTSTRAPS, permutation_samples=PERMUTATIONS)
    return reports

attack_results = {}
if RUN_MODEL_ATTACKS:
    for model in MODELS:
        split, members, validation, test = cohorts_for(model)
        engine = create_engine(repository_root=REPO_ROOT, model_root=MODEL_ROOT,
                               base_model_root=None, model=model, language=DATASET,
                               variant=VARIANT, device=DEVICE)
        member_scores = infer_scores(engine, members)
        validation_scores = infer_scores(engine, validation)
        test_scores = infer_scores(engine, test)
        attack_results[model] = {
            'selected_split': split,
            'validation_nonmember': compare_cohorts(member_scores, validation_scores),
            'test_nonmember_sensitivity': compare_cohorts(member_scores, test_scores),
            'member_accuracy': float(np.mean([row['correct'] for row in member_scores])),
            'validation_accuracy': float(np.mean([row['correct'] for row in validation_scores])),
            'test_accuracy': float(np.mean([row['correct'] for row in test_scores])),
        }
        del engine, member_scores, validation_scores, test_scores
        gc.collect()
else:
    print('Set RUN_MODEL_ATTACKS=1 and restart the kernel to run model queries.')

if attack_results:
    display(pd.DataFrame([
        {'model': model, 'nonmember': cohort, 'feature': feature,
         'auc': report['auc'], 'ci_low': report['auc_bootstrap_95_ci'][0],
         'ci_high': report['auc_bootstrap_95_ci'][1],
         'p': report['permutation_p_one_sided'],
         'max_advantage': report['maximum_tpr_minus_fpr']}
        for model, result in attack_results.items()
        for cohort in ('validation_nonmember', 'test_nonmember_sensitivity')
        for feature, report in result[cohort].items()
    ]))

Set RUN_MODEL_ATTACKS=1 and restart the kernel to run model queries.


## 4. Target-identity counterfactual

For unmasked models, replace every tagged target with a synthetic name while preserving the surrounding article. We report only prediction-flip rate and Jensen–Shannon divergence. This asks whether target identity acts as a memorized shortcut; it does not attempt to recover the original name. Masked models should be nearly invariant by construction.

In [5]:
ASPECT_RE = re.compile(r'<aspect>.*?</aspect>', flags=re.DOTALL)

def pseudonymize(row):
    clone = dict(row)
    clone['article'] = ASPECT_RE.sub('<aspect>Sintetični Cilj</aspect>', row['article'])
    clone['aspect'] = 'Sintetični Cilj'
    return clone

def js_divergence(left, right):
    left = np.clip(np.asarray(left, dtype=float), 1e-12, 1)
    right = np.clip(np.asarray(right, dtype=float), 1e-12, 1)
    middle = (left + right) / 2
    return float((np.sum(left * np.log(left / middle)) +
                  np.sum(right * np.log(right / middle))) / 2)

counterfactual_results = {}
if RUN_COUNTERFACTUALS:
    for model in MODELS:
        _, _, validation, _ = cohorts_for(model)
        engine = create_engine(repository_root=REPO_ROOT, model_root=MODEL_ROOT,
                               base_model_root=None, model=model, language=DATASET,
                               variant=VARIANT, device=DEVICE)
        original = engine.predict_batch(validation, batch_size=1 if engine.backend == 'han' else 8)
        changed = engine.predict_batch([pseudonymize(row) for row in validation],
                                       batch_size=1 if engine.backend == 'han' else 8)
        divergences, flips = [], []
        for before, after in zip(original, changed, strict=True):
            p = [before['class_probabilities'][CLASS_KEY[label]] for label in (-1, 0, 1)]
            q = [after['class_probabilities'][CLASS_KEY[label]] for label in (-1, 0, 1)]
            divergences.append(js_divergence(p, q))
            flips.append(before['predicted_sentiment'] != after['predicted_sentiment'])
        counterfactual_results[model] = {
            'n': len(validation), 'prediction_flip_rate': float(np.mean(flips)),
            'mean_js_divergence_nats': float(np.mean(divergences)),
            'p95_js_divergence_nats': float(np.quantile(divergences, 0.95)),
        }
        del engine, original, changed
        gc.collect()
else:
    print('Set RUN_COUNTERFACTUALS=1 and restart the kernel to run target ablations.')

if counterfactual_results:
    display(pd.DataFrame.from_dict(counterfactual_results, orient='index'))

Set RUN_COUNTERFACTUALS=1 and restart the kernel to run target ablations.


## 5. Aggregate-only export and release decision

The suggested review trigger below is deliberately conservative and is not a universal standard: investigate any fixed attack whose bootstrap lower bound exceeds 0.5 and whose AUC is at least 0.60, especially if replicated with both validation and test nonmembers. Also investigate large train/nonmember accuracy gaps or strong target-name sensitivity. Release approval remains a governance decision informed by these results, licensing, consent, and contractual constraints.

In [6]:
review_triggers = []
for model, result in attack_results.items():
    for cohort in ('validation_nonmember', 'test_nonmember_sensitivity'):
        for feature, report in result[cohort].items():
            if report['auc'] >= 0.60 and report['auc_bootstrap_95_ci'][0] > 0.50:
                review_triggers.append({'model': model, 'cohort': cohort,
                                        'feature': feature, 'auc': report['auc'],
                                        'ci': report['auc_bootstrap_95_ci']})

export = {
    'schema_version': 1,
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'dataset': DATASET, 'variant': VARIANT, 'models': MODELS,
    'cohort_cap': MAX_PER_COHORT, 'seed': SEED,
    'checkpoint_inventory': inventories,
    'membership_attacks': attack_results,
    'target_counterfactuals': counterfactual_results,
    'review_triggers': review_triggers,
    'limitations': [
        'No formal differential-privacy guarantee.',
        'Passing tested attacks does not rule out stronger membership or inversion attacks.',
        'Validation data influenced model selection.',
        'No raw text, aspect names, or per-record model outputs are exported.',
    ],
}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
report_path = OUTPUT_DIR / 'aggregate-privacy-audit.json'
report_path.write_text(json.dumps(export, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print({'report': str(report_path), 'review_trigger_count': len(review_triggers),
       'tensor_only': all(row['tensor_only'] for row in inventories)})

{'report': '/Utilisateurs/nchatt01/GitHub/aspect-based-sentiment-analysis/outputs/privacy-audit/hbs/20260908T143243Z/aggregate-privacy-audit.json', 'review_trigger_count': 0, 'tensor_only': True}


## Recommended follow-up

1. Run HBS on all three families and retain the aggregate report.
2. If authorized, run Slovenian internally with `ALLOW_RESTRICTED_SLOVENE=1`; do not commit its report.
3. Replicate any trigger with larger balanced cohorts and multiple seeds.
4. Add shadow-model or LiRA-style attacks if a stronger adversarial assessment is required. The three split checkpoints in the ignored training directories can support that work.
5. Exact canary exposure can only be measured properly when unique canaries were inserted before training; retrospective synthetic canaries do not test the completed checkpoints.
6. If Slovenian release is not approved, keep its checkpoint files only in the private release archive and remove them from the public Hugging Face repositories before changing repository visibility.